# Capstone — Refresh Opportunity Scoring

## 0. Abstract
This study asks which existing content pages should be reviewed first when editorial resources are limited. We use the bundled anonymized FlyRank content-refresh snapshot of 30,000 pages and define observed decline as `trend_direction == down`. A transparent refresh score is compared with Logistic Regression, Decision Tree and Random Forest models under a client-holdout evaluation with leakage checks. Random Forest achieved Precision@50 of 0.740 versus 0.240 for the baseline and ROC-AUC of 0.750 on the held-out test set. The output is a ranked, human-review decision-support queue rather than a causal claim about search-engine behavior.

## 1. Question
**Which existing pages should be reviewed first for refresh based on measurable signals associated with observed decline?**

## 2. Data
30,000 anonymized rows × 44 columns from the bundled content-refresh release. Rows with zero impressions or content age below 90 days would be excluded by the preparation contract; the current release retained all 30,000 rows after checks. IDs are grouping/traceability only.

## 3. Methodology
Target: `trend_direction == down`. Excluded leakage/identifier fields: `trend_direction`, `trend_pct`, `content_id`, `client_id`. Numeric and categorical page/performance descriptors are modeled. Baseline: 40% visibility + 30% freshness risk + 25% position opportunity + 5% depth gap. Models: Logistic Regression, Decision Tree, Random Forest. Split: client holdout, approximately 20% of clients reserved for testing. Primary metric: Precision@50.

In [ ]:
import json
from pathlib import Path
ROOT=Path.cwd()
while ROOT.name and not (ROOT/'outputs'/'model_results.json').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
r=json.loads((ROOT/'outputs'/'model_results.json').read_text())
print('Best:',r['best_model']['name'])
print('Base rate:',r['target_positive_rate'])
print('RF:',r['models']['random_forest'])
print('Baseline:',r['baseline'])

## 4. Results
| Method | ROC-AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| Baseline rules | 0.627 | 0.468 | 0.240 | — | — |
| Logistic Regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| Decision Tree | 0.742 | 0.575 | 0.660 | 0.716 | 0.634 |
| Random Forest | **0.750** | **0.618** | **0.740** | 0.744 | **0.640** |

## 5. Limitations
The snapshot does not provide a genuine future-window outcome, so this should not be treated as a temporal forecast. The label is observational. Feature importance does not establish causation. Results may change on another release, client mix or distribution.

## 6. Ranked recommendations
1. Review high-confidence declining pages first.
2. For visible pages with low CTR, inspect title/snippet/search-intent alignment manually.
3. For low-engagement candidates, inspect content usefulness and page experience.
4. Treat stale-age signals as supporting evidence only because the simple staleness flag was not confirmed in the signal audit.
5. Keep monitor items out of immediate refresh work unless editorial context justifies them.

## 7. Artifacts
Use `outputs/refresh_queue.csv`, `outputs/model_results.json`, and the charts under `outputs/charts/`. The queue contains reason codes, confidence and suggested actions.

## 8. Reproducibility
From the repository root: `pip install -r requirements.txt` then `python scripts/run_all.py`. Random state is 42. The raw starter CSV is public-safe and read-only.

## 9. Acknowledgments & data credit
Built on the FlyRank ML Internship dataset — https://flyrank.ai

## ML-12 — 5-minute demo
1. Problem and decision (45s)
2. Data and leakage controls (45s)
3. Baseline vs Random Forest (60s)
4. Ranked queue + reason codes (90s)
5. Limitations and next step (60s)

## Social-post cut
Built a refresh-prioritization model on the FlyRank internship dataset. Random Forest reached 0.740 Precision@50 vs 0.240 for the transparent baseline under a client-holdout test. The output is decision support for human review—not a claim about Google or causality.

## Employer-facing summary
I built an end-to-end content refresh scoring pipeline from data contract and leakage audit through baseline, model comparison, grouped validation and an action queue. The best model improved Precision@50 from 0.240 to 0.740 on the held-out client set. I packaged the result as a reproducible, public-safe research artifact with explicit limitations.